# pi-eval on Kaggle (2x T4) — Qwen3-4B Thinking (2507)

This notebook runs the pi-eval harness inside Kaggle against **qwen3:4b-thinking-2507** served by Ollama at its **maximum context window (262,144 tokens)**. Ollama's OpenAI-compatible endpoint at `http://localhost:11434/v1` is used as the eval provider (`--api-base`).

After the eval run, the notebook pushes the evaluated model to HuggingFace so you have the exact model that ran the eval. Replace HF_REPO in the push cell and run huggingface-cli login (or set HF_TOKEN) first.


**Before running:**
1. Kaggle UI: set **Accelerator -> GPU T4 x2**.
2. Kaggle UI: enable **Internet** (needed for apt/npm/pip and for Ollama).
3. Replace `REPO_URL` below with your fork's URL, or upload the repo as a Kaggle Dataset and use the copy cell.

In [ ]:
!nvidia-smi
!free -h
!nproc
!df -h /kaggle/working

In [ ]:
REPO_URL = 'https://github.com/<your-user>/pi-eval.git'
MODEL_ID = 'qwen3:4b-thinking-2507-max'  # qwen3:4b-thinking-2507 at maximum context
MAX_CTX = 262144                          # native max context of the 2507 refresh
PORT = 11434
RUNS = 5
SUITE = 'extraction'
print(MODEL_ID, MAX_CTX, SUITE, RUNS)

## Get the code

Clone from GitHub, or if you uploaded the repo as a Kaggle Dataset, run the commented copy cell instead.

In [ ]:
!git clone {REPO_URL} /kaggle/working/pi-eval
# !cp -r /kaggle/input/pi-eval /kaggle/working/pi-eval

## Install Node 22+ and the repo dependencies

The harness needs Node 22.19+. Model catalog data is gitignored, so hydrate it and build the workspace packages once.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
!apt-get install -y nodejs
!node --version && npm --version

In [ ]:
%cd /kaggle/working/pi-eval
!npm ci --ignore-scripts
!npm run hydrate:model-data
!npm run build:offline

In [ ]:
!pip -q install requests pandas huggingface_hub

## Install Ollama and pull the model

Install the official Ollama CLI, then pull `qwen3:4b-thinking-2507` (the 2507 refresh).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
!ollama pull qwen3:4b-thinking-2507
!ollama show qwen3:4b-thinking-2507 --modelfile

## Create the maximum-context model

Ollama serves models with the `num_ctx` parameter; the 2507 refresh ships 262,144 as its native maximum. We pin `num_ctx` to that maximum in a Modelfile and create `qwen3:4b-thinking-2507-max` from it. (To serve the base tag with the same maximum instead, start the server with `OLLAMA_CONTEXT_LENGTH=262144`.)

In [ ]:
modelfile = f"FROM qwen3:4b-thinking-2507\nPARAMETER num_ctx {MAX_CTX}\n"
with open('/kaggle/working/Modelfile', 'w') as f:
    f.write(modelfile)
print(modelfile)

!ollama create qwen3:4b-thinking-2507-max -f /kaggle/working/Modelfile
!ollama show qwen3:4b-thinking-2507-max --modelfile | grep -i -E 'FROM|num_ctx'

## Serve the model

Start `ollama serve` in the background on `localhost:11434` and wait until the API answers.

In [ ]:
import subprocess
import time

import requests

log = open('/kaggle/working/ollama.log', 'w')
server = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT)
print('ollama serve pid', server.pid)

ready = False
for _ in range(60):
    try:
        if requests.get(f'http://localhost:{PORT}/api/tags', timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    server.terminate()
    log.close()
    raise RuntimeError('ollama serve did not become ready; see /kaggle/working/ollama.log')

tags = requests.get(f'http://localhost:{PORT}/api/tags', timeout=2).json()
print('serving:', [m['name'] for m in tags.get('models', [])])

## Run the eval

First a one-task smoke run to prove the endpoint works, then the full suite with repetitions.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/pi-eval/packages/evals/notebooks')
from pi_eval import PiEval

client = PiEval(repo_path='/kaggle/working/pi-eval')
smoke = client.run(
    suite=SUITE,
    tasks=['oven-payment-real'],
    runs=1,
    backend='host',
    model=MODEL_ID,
    api_base=f'http://localhost:{PORT}/v1',
)
print('ok:', smoke.ok, 'traces:', len(smoke.traces))
smoke.report

In [ ]:
result = client.run(
    suite=SUITE,
    runs=RUNS,
    backend='host',
    model=MODEL_ID,
    api_base=f'http://localhost:{PORT}/v1',
)
print('ok:', result.ok)
result.save('/kaggle/working/pi-eval-run')
result.report

In [ ]:
result.traces_frame().head(20)
print('datasets:', list(result.datasets))

## Push the eval model to HuggingFace

Ollama does not push models to HuggingFace (it has its own registry), so we export the GGUF blob of the model used in the eval and upload it to the HF Hub. First run huggingface-cli login (or set HF_TOKEN), and replace HF_REPO with your own.


In [ ]:
import glob
import json
import os
import shutil

from huggingface_hub import HfApi, upload_file

HF_REPO = '<your-hf-username>/qwen3-4b-thinking-2507-eval'

root = os.path.expanduser('~/.ollama/models')
repo, _, tag = MODEL_ID.partition(':')
manifests = glob.glob(os.path.join(root, 'manifests', '**', repo, tag), recursive=True)
layers = [
    layer
    for path in manifests
    for layer in json.load(open(path)).get('layers', [])
    if layer.get('mediaType') == 'application/vnd.ollama.image.model'
]
if not layers:
    raise RuntimeError('no GGUF layer for ' + MODEL_ID)

blob = os.path.join(root, 'blobs', layers[0]['digest'].replace(':', '-'))
out = '/kaggle/working/Qwen3-4B-Thinking-2507-Max.gguf'
shutil.copy(blob, out)
print('exported', layers[0].get('size'), 'bytes to', out)

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, private=False)
token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
upload_file(path_or_fileobj=out, path_in_repo='Qwen3-4B-Thinking-2507-Max.gguf', repo_id=HF_REPO, token=token)
print('pushed to https://huggingface.co/' + HF_REPO)


Other suites you can switch to in the constants cell above: `docs`, `example`, `data-manipulation-read-organize`, `extraction-minimal`.

In [ ]:
# Stop the model server when finished
server.terminate()
log.close()